# Inventory Diff and Reference Generation

Runs inventory change detection and per-object reference generation. 

In [ ]:
import sys
import warnings
from pathlib import Path
from datetime import datetime, UTC
import pandas as pd

repo_root = Path('..').resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from utils.config_utils import check_runtime_readiness, load_pipeline_config, resolve_secrets
from pipeline.inventory import build_inventory_snapshot_and_diff
from pipeline.generate_parquet import concurrent_dask_ref_generation, save_ledger_after_success

print('Imports OK')

In [ ]:
check_runtime_readiness()
kp = load_pipeline_config("../configs/config.yaml")
ACCESS_KEY, SECRET_KEY = resolve_secrets(kp)

warnings.filterwarnings(
    "ignore",
    message="Numcodecs codecs are not in the Zarr version 3 specification*",
    category=UserWarning
)

## MasterLedger building

In [ ]:
ledger = build_inventory_snapshot_and_diff(
    kp=kp,
    access_key=ACCESS_KEY,
    secret_key=SECRET_KEY,
)

print('Inventory summary:', ledger['summary'])

inventory_diff = ledger['diff']
inventory_objects = ledger['current_objects']
pending_ledger = ledger['next_ledger']

## Reference Generation (Full Run)

Processes all new/changed objects concurrently via Dask.

In [ ]:
full_start_time = datetime.now(UTC)

reference_generation = concurrent_dask_ref_generation(
    kp=kp,
    access_key=ACCESS_KEY,
    secret_key=SECRET_KEY,
    inventory_diff=inventory_diff,
    current_objects=inventory_objects,
)

full_end_time = datetime.now(UTC)
gen_duration = str(full_end_time - full_start_time).split(".")[0]

print("Reference generation summary:", reference_generation["summary"])
print("Full-run duration:", gen_duration)

## Commit Ledger and Log Runtime

In [ ]:
from uuid import uuid4

runtime_log_path = Path(".runtime_logs/full_run.csv")
runtime_log_path.parent.mkdir(exist_ok=True)

commit_start = datetime.now(UTC)
if reference_generation["summary"]["failed"] == 0:
    save_ledger_after_success(
        ledger_path=kp["output"]["ledger_path"],
        next_ledger=pending_ledger,
        generation_summary=reference_generation["summary"],
    )
    commit_status = "committed"
    print("Ledger committed:", kp["output"]["ledger_path"])
else:
    commit_status = "blocked_due_to_failures"
    print("Ledger NOT committed due to generation failures.")

commit_end = datetime.now(UTC)
commit_ledger_time = str(commit_end - commit_start).split(".")[0]

runtime_log_row = {
    "run_id": str(uuid4()),
    "start_time": full_start_time,
    "ended_at": commit_end,
    "gen_duration": gen_duration,
    "commit_ledger_time": commit_ledger_time,
    "scanned": int(reference_generation["summary"].get("scanned", 0)),
    "changed_or_new": int(reference_generation["summary"].get("changed_or_new", 0)),
    "generated": int(reference_generation["summary"].get("generated", 0)),
    "failed": int(reference_generation["summary"].get("failed", 0)),
    "commit_status": commit_status,
}

df = pd.DataFrame([runtime_log_row])
header = not runtime_log_path.exists()
df.to_csv(runtime_log_path, mode='a', index=False, header=header)

print(f"Runtime log appended to: {runtime_log_path}")
print("Full-run duration:", gen_duration)
print("Commit ledger time:", commit_ledger_time)